# Mercedes F1 Infringement Profile - LexRank Summarization (Version 2)

This notebook implements LexRank algorithm for extractive summarization of Mercedes F1 infringement documents with modified parameters.

## Objective:
- Process all `no_footer_` files from the preprocessed dataset
- Apply LexRank algorithm for extractive summarization with updated parameters
- Maintain temporal analysis (year-by-year summaries)
- Generate "Version Two" of the infringement profile using LexRank

## Input:
- `pre_proc_op/` folder containing `no_footer_*.txt` files organized by year

## Output:
- Console summaries for each year
- Overall consolidated summary
- Results saved to `lexrank_results_v2/` folder


In [1]:
# Import required libraries
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# LexRank implementation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("Libraries imported successfully")


Libraries imported successfully


In [2]:
# Configuration
processed_base_path = Path("pre_proc_op")
years = ["2020", "2021", "2022", "2023", "2024"]

# LexRank parameters (Version 2 - Modified)
SENTENCE_COUNT = 8  # Number of top sentences to extract (increased from 5)
SIMILARITY_THRESHOLD = 0.15  # Minimum similarity threshold for graph edges (increased from 0.1)

print("Configuration (Version 2):")
print(f"Processed base path: {processed_base_path}")
print(f"Years to analyze: {years}")
print(f"Sentences per summary: {SENTENCE_COUNT}")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")


Configuration (Version 2):
Processed base path: pre_proc_op
Years to analyze: ['2020', '2021', '2022', '2023', '2024']
Sentences per summary: 8
Similarity threshold: 0.15


In [3]:
# LexRank implementation
class LexRankSummarizer:
    def __init__(self, sentence_count=5, similarity_threshold=0.1):
        self.sentence_count = sentence_count
        self.similarity_threshold = similarity_threshold
        self.stemmer = PorterStemmer()
        
        # Get stopwords
        try:
            self.stop_words = set(stopwords.words('english'))
        except LookupError:
            nltk.download('stopwords')
            self.stop_words = set(stopwords.words('english'))
    
    def preprocess_sentence(self, sentence):
        """Preprocess sentence for similarity calculation"""
        # Convert to lowercase
        sentence = sentence.lower()
        
        # Remove special characters and digits
        sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)
        
        # Tokenize and remove stopwords
        words = word_tokenize(sentence)
        words = [self.stemmer.stem(word) for word in words if word not in self.stop_words]
        
        return ' '.join(words)
    
    def build_similarity_matrix(self, sentences):
        """Build similarity matrix using TF-IDF and cosine similarity"""
        n = len(sentences)
        
        # Preprocess all sentences
        processed_sentences = [self.preprocess_sentence(sent) for sent in sentences]
        
        # Create TF-IDF vectors
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(processed_sentences)
        
        # Calculate cosine similarity matrix
        similarity_matrix = cosine_similarity(tfidf_matrix)
        
        # Apply threshold to create binary adjacency matrix
        adjacency_matrix = (similarity_matrix >= self.similarity_threshold).astype(int)
        
        # Remove self-loops
        np.fill_diagonal(adjacency_matrix, 0)
        
        return adjacency_matrix, similarity_matrix
    
    def calculate_centrality_scores(self, adjacency_matrix):
        """Calculate LexRank centrality scores using power iteration"""
        n = adjacency_matrix.shape[0]
        
        # Initialize scores uniformly
        scores = np.ones(n) / n
        
        # Power iteration parameters
        max_iterations = 100
        tolerance = 1e-6
        
        for iteration in range(max_iterations):
            old_scores = scores.copy()
            
            # Calculate new scores
            for i in range(n):
                # Sum of scores of connected sentences
                connected_sum = np.sum(adjacency_matrix[i] * scores)
                # Degree of current sentence
                degree = np.sum(adjacency_matrix[i])
                
                if degree > 0:
                    scores[i] = connected_sum / degree
                else:
                    scores[i] = 0
            
            # Check for convergence
            if np.max(np.abs(scores - old_scores)) < tolerance:
                break
        
        return scores
    
    def select_diverse_sentences(self, sentences, scores, similarity_matrix):
        """Select diverse sentences to avoid redundancy"""
        selected_indices = []
        remaining_indices = list(range(len(sentences)))
        
        # Sort by centrality scores
        sorted_indices = sorted(remaining_indices, key=lambda x: scores[x], reverse=True)
        
        for idx in sorted_indices:
            if len(selected_indices) >= self.sentence_count:
                break
            
            # Check if this sentence is too similar to already selected ones
            is_diverse = True
            for selected_idx in selected_indices:
                if similarity_matrix[idx][selected_idx] > 0.7:  # High similarity threshold
                    is_diverse = False
                    break
            
            if is_diverse:
                selected_indices.append(idx)
        
        # Sort selected indices by original order
        selected_indices.sort()
        
        return selected_indices
    
    def summarize(self, text):
        """Generate summary using LexRank algorithm"""
        if not text or len(text.strip()) < 100:
            return "Insufficient text for summarization."
        
        # Tokenize into sentences
        sentences = sent_tokenize(text)
        
        if len(sentences) < 2:
            return text
        
        # Build similarity matrix
        adjacency_matrix, similarity_matrix = self.build_similarity_matrix(sentences)
        
        # Calculate centrality scores
        centrality_scores = self.calculate_centrality_scores(adjacency_matrix)
        
        # Select diverse sentences
        selected_indices = self.select_diverse_sentences(sentences, centrality_scores, similarity_matrix)
        
        # Extract selected sentences
        summary_sentences = [sentences[idx] for idx in selected_indices]
        
        return ' '.join(summary_sentences)

print("LexRankSummarizer class defined successfully")


LexRankSummarizer class defined successfully


In [4]:
# Initialize LexRank summarizer
summarizer = LexRankSummarizer(sentence_count=SENTENCE_COUNT, similarity_threshold=SIMILARITY_THRESHOLD)

print("LexRank summarizer initialized (Version 2)")
print(f"Parameters: {SENTENCE_COUNT} sentences, {SIMILARITY_THRESHOLD} similarity threshold")


LexRank summarizer initialized (Version 2)
Parameters: 8 sentences, 0.15 similarity threshold


In [5]:
# Process documents year by year
print("MERCEDES F1 INFRINGEMENT PROFILE - LEXRANK SUMMARIZATION (VERSION 2)")

yearly_summaries = {}
yearly_stats = {}

for year in years:
    print(f"\nPROCESSING {year} - MERCEDES F1 INFRINGEMENTS")
    
    year_path = processed_base_path / year
    
    if not year_path.exists():
        print(f"Folder {year_path} does not exist")
        continue
    
    # Get all no_footer_ files
    no_footer_files = list(year_path.glob("no_footer_*.txt"))
    
    if not no_footer_files:
        print(f"No no_footer_ files found in {year}")
        continue
    
    print(f"Found {len(no_footer_files)} processed documents in {year}")
    
    # Read and combine all documents for the year
    combined_text = ""
    total_chars = 0
    processed_files = 0
    
    for file_path in no_footer_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if content:
                    combined_text += content + " "
                    total_chars += len(content)
                    processed_files += 1
        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")
    
    if not combined_text.strip():
        print(f"No valid content found in {year}")
        continue
    
    print(f"Processed {processed_files} files, {total_chars:,} total characters")
    
    # Generate summary using LexRank
    print(f"\nGenerating LexRank summary for {year}...")
    summary = summarizer.summarize(combined_text)
    
    # Store results
    yearly_summaries[year] = summary
    yearly_stats[year] = {
        'files_processed': processed_files,
        'total_chars': total_chars,
        'summary_length': len(summary)
    }
    
    # Display summary
    print(f"\nLEXRANK SUMMARY FOR {year}:")
    print("-" * 50)
    print(summary)
    print("-" * 50)
    print(f"Summary length: {len(summary):,} characters")
    print(f"Compression ratio: {(len(summary)/total_chars)*100:.1f}%")

print(f"\nProcessed {len(yearly_summaries)} years successfully")


MERCEDES F1 INFRINGEMENT PROFILE - LEXRANK SUMMARIZATION (VERSION 2)

PROCESSING 2020 - MERCEDES F1 INFRINGEMENTS
Found 11 processed documents in 2020
Processed 11 files, 10,876 total characters

Generating LexRank summary for 2020...

LEXRANK SUMMARY FOR 2020:
--------------------------------------------------
Offence Alleged breach of Appendix H Article 2.5.5.1.b) of the FIA International Sporting Code. Decision No further action. The Stewards received a petition from Aston Martin Red Bull Racing (including video footage) for them to review, in accordance with Article 14 of the FIA International Sporting Code, the following decision made by the Stewards at the 2020 Austrian Grand Prix Competition: Document 33 – 2020 Austrian Grand Prix. The Stewards, summoned the team (Document 39) and held a hearing at 13:45 on Sunday 5th July, 2020. The Stewards determine that the additional video evidence represents a significant and relevant new element which was unavailable to the parties at the

In [6]:
# Generate overall consolidated summary
print("\nOVERALL CONSOLIDATED SUMMARY - ALL YEARS")

if yearly_summaries:
    # Combine all yearly summaries
    all_summaries_text = " "
    for year, summary in yearly_summaries.items():
        all_summaries_text += f"{year} Summary: {summary} "
    
    # Generate overall summary
    print("Generating overall consolidated summary...")
    overall_summary = summarizer.summarize(all_summaries_text)
    
    print("\nOVERALL MERCEDES F1 INFRINGEMENT PROFILE (LEXRANK V2):")
    print("=" * 70)
    print(overall_summary)
    print("=" * 70)
    
    # Statistics
    total_files = sum(stats['files_processed'] for stats in yearly_stats.values())
    total_chars = sum(stats['total_chars'] for stats in yearly_stats.values())
    
    print(f"\nOVERALL STATISTICS:")
    print(f"Total documents processed: {total_files}")
    print(f"Total characters analyzed: {total_chars:,}")
    print(f"Overall summary length: {len(overall_summary):,} characters")
    print(f"Overall compression ratio: {(len(overall_summary)/total_chars)*100:.1f}%")
    
    print(f"\nYEARLY BREAKDOWN:")
    for year, stats in yearly_stats.items():
        compression = (stats['summary_length']/stats['total_chars'])*100 if stats['total_chars'] > 0 else 0
        print(f"{year}: {stats['files_processed']} docs, {stats['total_chars']:,} chars -> {stats['summary_length']:,} chars ({compression:.1f}%)")
    
else:
    print("No summaries generated. Please check the input files.")



OVERALL CONSOLIDATED SUMMARY - ALL YEARS
Generating overall consolidated summary...

OVERALL MERCEDES F1 INFRINGEMENT PROFILE (LEXRANK V2):
Decision No further action. The Stewards received a petition from Aston Martin Red Bull Racing (including video footage) for them to review, in accordance with Article 14 of the FIA International Sporting Code, the following decision made by the Stewards at the 2020 Austrian Grand Prix Competition: Document 33 – 2020 Austrian Grand Prix. Reasons: During the initial hearing, no on-board video footage of car 44 was available to the Stewards. (2 penalty points awarded, 5 points in total for the 12 month period) Reason Following a petition to review decision 33 taken after Qualifying, the Stewards acknowledged that the on-board footage of car 44 represents a significant new element (see decision 41) that had not been available to the Stewards in the hearing on Saturday and therefore reviewed the case. 12.4.1 m of the FIA International Sporting Code) R

In [7]:
# Save results to files
print("\nSAVING RESULTS")

# Create output directory (Version 2)
output_dir = Path("lexrank_results_v2")
output_dir.mkdir(exist_ok=True)

# Save yearly summaries
for year, summary in yearly_summaries.items():
    output_file = output_dir / f"lexrank_summary_{year}.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Mercedes F1 Infringement Summary - {year}\n")
        f.write("="*50 + "\n\n")
        f.write(summary)
    print(f"Saved {year} summary to {output_file}")

# Save overall summary
if yearly_summaries:
    overall_file = output_dir / "lexrank_overall_summary.txt"
    with open(overall_file, 'w', encoding='utf-8') as f:
        f.write("Mercedes F1 Infringement Profile - Overall Summary (LexRank V2)\n")
        f.write("="*70 + "\n\n")
        f.write(overall_summary)
        f.write("\n\n" + "="*70 + "\n")
        f.write("STATISTICS:\n")
        f.write(f"Total documents processed: {total_files}\n")
        f.write(f"Total characters analyzed: {total_chars:,}\n")
        f.write(f"Overall summary length: {len(overall_summary):,} characters\n")
        f.write(f"Overall compression ratio: {(len(overall_summary)/total_chars)*100:.1f}%\n")
    print(f"Saved overall summary to {overall_file}")

print(f"\nLexRank summarization (Version 2) completed successfully!")
print(f"Results saved in: {output_dir}")



SAVING RESULTS
Saved 2020 summary to lexrank_results_v2\lexrank_summary_2020.txt
Saved 2021 summary to lexrank_results_v2\lexrank_summary_2021.txt
Saved 2022 summary to lexrank_results_v2\lexrank_summary_2022.txt
Saved 2023 summary to lexrank_results_v2\lexrank_summary_2023.txt
Saved 2024 summary to lexrank_results_v2\lexrank_summary_2024.txt
Saved overall summary to lexrank_results_v2\lexrank_overall_summary.txt

LexRank summarization (Version 2) completed successfully!
Results saved in: lexrank_results_v2
